# M3 IA Agente - Fluxo de trabalho de assistente de e-mail

## 1. Introdução

### 1.1 Visão geral do laboratório

Em um vídeo anterior deste módulo, Andrew mostrou o exemplo de um assistente de calendário que podia usar várias ferramentas para realizar tarefas complexas. Neste laboratório não avaliado, você explorará um exemplo semelhante: trabalhar com um **agente assistente de e-mail**.

Este fluxo de trabalho agente pode realizar várias tarefas relacionadas ao gerenciamento de e-mails, incluindo o envio de e-mails, a busca por e-mails de um remetente específico e a exclusão de e-mails. Você dará instruções em linguagem natural, como "verificar e-mails não lidos do meu chefe" ou "excluir o e-mail do happy hour", e verá como ele seleciona as ferramentas certas e conclui a tarefa para você.

<img src="lab_overview.png" alt="Exemplo de um assistente de calendário" width="700"/>

### 🎯 1.2 Resultado de aprendizagem
Ao final deste laboratório, você será capaz de **conectar um LLM a ferramentas** com o AISuite, fornecer instruções em linguagem natural e observar como o agente seleciona, executa e valida tarefas de várias etapas, como pesquisar, enviar e excluir e-mails.

## 2. Inicializar o ambiente e o cliente

Assim como no laboratório anterior, você configurará o ambiente.

In [ ]:
# ================================
# Imports
# ================================

# --- Third-party ---
from dotenv import load_dotenv
import aisuite as ai
import json

# --- Local / project ---
import utils
import display_functions
import email_tools



# ================================
# Environment & Client
# ================================
load_dotenv()          # Load environment variables from .env
client = ai.Client()   # Initialize AISuite client


## 3. Serviço de e-mail simulado

### 3.1 Componentes
Este laboratório utiliza um **backend de e-mail simulado** para imitar interações de e-mail do mundo real.

Pense nele como sua caixa de entrada de e-mail pessoal para testes: ela vem pré-carregada com mensagens para que você possa praticar sem enviar e-mails reais.

Em vez de construir este backend do zero, você usará ferramentas que permitem interagir diretamente com ele. Nos bastidores, ele utiliza:

| Camada | Finalidade |
|-------------------------|--------------------------------|
| **FastAPI** | Expõe endpoints REST |
| **SQLite + SQLAlchemy** | Armazena e consulta e-mails localmente |
| **Pydantic** | Garante que as entradas e saídas sejam válidas |
| **Ferramentas AISuite** | Ponte entre o LLM e o serviço |

### 3.2 Endpoints
O serviço fornece diversas rotas que simulam ações comuns de e-mail.

Mais tarde, essas funções serão encapsuladas como ferramentas para que o assistente possa usá-las automaticamente:

- `POST /send` → enviar um novo e-mail
- `GET /emails` → listar todos os e-mails
- `GET /emails/unread` → mostrar apenas os e-mails não lidos
- `GET /emails/{id}` → buscar um e-mail específico pelo ID
- `GET /emails/search?q=...` → pesquisar e-mails por palavra-chave
- `GET /emails/filter` → filtrar por destinatário ou intervalo de datas
- `PATCH /emails/{id}/read` → marcar um e-mail como lido
- `PATCH /emails/{id}/unread` → marcar um e-mail como não lido
- `DELETE /emails/{id}` → excluir um e-mail pelo ID
- `GET /reset_database` → redefinir os e-mails para o estado inicial (para testes)

> 💡 **Ideia principal:** Esses endpoints atuam como os “controles da caixa de entrada”.

Nos próximos passos, você os exporá como funções Python (ferramentas) que o LLM pode chamar — transformando rotas brutas em ações do agente.

### 3.3 Auxiliares de teste de endpoint
Antes de dar o controle ao LLM, você primeiro **testará o backend por conta própria**.

As funções `utils.test_*` são wrappers rápidos em torno dos endpoints da API. Elas permitem que você experimente ações como **enviar, listar, pesquisar, filtrar, marcar e excluir** sem escrever solicitações HTTP brutas.

**Observação:** encontre o arquivo utils.py no menu superior navegando até Arquivo > Abrir.

Cada auxiliar tem um nome claro e autoexplicativo. No próximo trecho de código, você pode descomentar o que deseja executar, verificar a saída e confirmar se o serviço de e-mail se comporta conforme o esperado.

Por exemplo, você pode testar:
- Enviar um e-mail de teste
- Buscar e-mails por ID
- Listar todas as mensagens
- Marcar um e-mail como lido ou não lido
- Excluir um e-mail

Esta etapa serve como uma **verificação de integridade** antes de entregar o controle ao agente. Ela garante que o serviço de e-mail simulado esteja funcionando corretamente e se comportando exatamente como um serviço de e-mail real.

👉 Para testar os endpoints, **descomente** as linhas que deseja testar, execute a célula e verifique os resultados instantaneamente.

In [ ]:
# uncomment the line 'utils.test_*' you want to try
new_email_id = utils.test_send_email()
_ = utils.test_get_email(new_email_id['id'])
#_ = utils.test_list_emails()
#_ = utils.test_filter_emails(recipient="test@example.com")
#_ = utils.test_search_emails("lunch")
#_ = utils.test_unread_emails()
#_ = utils.test_mark_read(new_email_id['id'])
#_ = utils.test_mark_unread(new_email_id['id'])
#_ = utils.test_delete_email(new_email_id['id'])
#_ = utils.reset_database()


## 4. Camada de ferramentas para o agente de e-mail

### 4.1 Por que ferramentas?

Agora que os endpoints estão funcionando, o próximo passo é expô-los ao LLM como funções Python chamadas **ferramentas**. Cada ferramenta encapsula uma rota REST, transformando chamadas brutas à API em ações que o agente pode executar — como listar, ler, pesquisar, enviar, excluir ou alternar a leitura.

> Pense nas ferramentas como os **atuadores** do agente: você dá uma instrução em linguagem natural (“verifique os e-mails não lidos do meu chefe e envie uma resposta educada”), e o modelo escolhe **quais ferramentas** chamar e **em que ordem** concluir a tarefa.

### 4.2 Dicas de design
- Mantenha as **docstrings** das ferramentas curtas, imperativas e específicas para a ação.

- Retorne **JSON consistente e compacto** para que o modelo possa encadear resultados.

- Prefira **uma responsabilidade clara por ferramenta** (rota única, efeito único).

### 4.3 Ferramentas disponíveis

| Função | Ação |
|------------------------------------|------------------------------------------------------------------------  |
| `list_all_emails()`                | Busca todos os e-mails, do mais recente para o mais antigo               |
| `list_unread_emails()`             | Recupera apenas os e-mails não lidos                                     |
| `search_emails(query)`             | Busca por palavra-chave no assunto, corpo ou remetente                   |
| `filter_emails(...)`               | Filtra por destinatário e/ou intervalo de datas                          |
| `get_email(email_id)`              | Busca um e-mail específico pelo ID                                       |
| `mark_email_as_read(id)`           | Marca um e-mail como lido                                                |
| `mark_email_as_unread(id)`         | Marca um e-mail como não lido                                            |
| `send_email(...)`                  | Envia um novo e-mail (simulado)                                          |
| `delete_email(id)`                 | Exclui um e-mail pelo ID                                                 |
| `search_unread_from_sender(addr)`  | Retornar e-mails não lidos de um remetente específico (por exemplo, `boss@email.com`) |

**Observação:** encontre o arquivo email_tools.py no menu superior, navegando até Arquivo > Abrir.

Por exemplo, **testando a ferramenta `list_all_emails()`:**
```python
all_emails = email_tools.list_all_emails()
```

Agora, vamos testar algumas ferramentas que se conectam aos endpoints simulados para garantir que tudo esteja funcionando.

👉 **Descomente** as que você deseja executar, execute a célula e revise os resultados.

In [ ]:
# Test sending a new email and fetch it by ID
new_email = email_tools.send_email("test@example.com", "Lunch plans", "Shall we meet at noon?")
content_ = email_tools.get_email(new_email['id'])

# Uncomment the ones you want to try:
#content_ = email_tools.list_all_emails()
#content_ = email_tools.list_unread_emails()
#content_ = email_tools.search_emails("lunch")
#content_ = email_tools.filter_emails(recipient="test@example.com")
#content_ = email_tools.mark_email_as_read(new_email['id'])
#content_ = email_tools.mark_email_as_unread(new_email['id'])
#content_ = email_tools.search_unread_from_sender("test@example.com")
#content_ = email_tools.delete_email(new_email['id'])

utils.print_html(content=json.dumps(content_, indent=2), title="Testing the email_tools")


## 5. Preparando o prompt do agente

Antes de atribuir tarefas ao agente assistente de e-mail, você criará uma pequena função auxiliar chamada `build_prompt()`.

Essa função envolve a solicitação em linguagem natural em um preâmbulo no estilo do sistema para que o LLM:

- Reconheça que está atuando como um **agente assistente de e-mail**
- Entenda que tem permissão para usar as ferramentas disponíveis
- Execute ações diretamente, sem pedir confirmação (sem intervenção humana)

In [ ]:
def build_prompt(request_: str) -> str:
    return f"""
- You are an AI assistant specialized in managing emails.
- You can perform various actions such as listing, searching, filtering, and manipulating emails.
- Use the provided tools to interact with the email system.
- Never ask the user for confirmation before performing an action.
- If needed, my email address is "you@email.com" so you can use it to send emails or perform actions related to my account.

{request_.strip()}
"""

Execute a próxima célula para ver como a função anterior **envolve seu prompt de usuário bruto** com instruções do sistema.
Por exemplo:

In [ ]:
example_prompt = build_prompt("Delete the Happy Hour email")
utils.print_html(content=example_prompt, title="Example example_prompt")


### 5.3 Reiniciando o serviço de e-mail

Como você estava experimentando o serviço de e-mail, vamos reiniciá-lo para o estado inicial.

Você pode fazer isso chamando a função utilitária que limpa e atualiza o serviço de e-mail simulado:

In [ ]:
utils.reset_database()

## 6. LLM + Ferramentas de E-mail

### 6.1 Cenário
Até agora, você trabalhou diretamente com o backend. Agora é hora de deixar o **LLM assumir o controle** como seu assistente de e-mail.

Por exemplo, você pode instruí-lo:
> “Verifique se há e-mails não lidos de `boss@email.com`, marque-os como lidos e envie uma mensagem de acompanhamento educada.”

### 6.2 O que acontece
1. O agente interpreta sua instrução.

2. Ele seleciona as ferramentas corretas (`search_unread_from_sender` → `mark_email_as_read` → `send_email`).

3. Ele executa cada ação automaticamente, sem pedir confirmação.

O AISuite lida com a exposição do esquema, a vinculação de argumentos, a execução e a passagem de resultados entre as etapas — para que você possa se concentrar no **que** o agente realiza, e não em **como** chamar a API.

### 6.3 Execute o teste
Execute a próxima célula e veja como o agente orquestra várias ferramentas para atender à sua solicitação. Sinta-se à vontade para testar suas próprias solicitações com o serviço de e-mail também.

*O que observar:*
- Um **rastreamento de chamadas de ferramentas** claro, mostrando quais ferramentas foram usadas e os argumentos passados.

- Uma **mensagem final** concisa, resumindo as ações realizadas (por exemplo, “Encontrado 1 e-mail não lido, marcado como lido, enviado acompanhamento”).

In [ ]:
# Try your own requests
prompt_ = build_prompt("Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.")

response = client.chat.completions.create(
    model="openai:gpt-4.1", # LLM
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # list of tools that the LLM can access
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)



### 6.4. Ferramenta ausente: `delete_email`

O que acontece se a ferramenta necessária não estiver disponível?

Por exemplo, tente esta solicitação:
> “Excluir o e-mail de alice@work.com.”

Como a ferramenta `delete_email` não está registrada, o LLM ainda tentará responder, mas não conseguirá concluir a ação.

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">

<h4>🔍 Insights importantes</h4>

<p style="margin: 0;">

Isso destaca um ponto importante: <b>as ferramentas que você disponibiliza determinam diretamente o que o agente pode fazer.</b>

</p>
</div>

**Ferramentas disponíveis**

| Ferramenta Função | Ação |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()` | Recuperar todos os emails, do mais recente para o mais antigo |
| `list_unread_emails()` | Recuperar apenas emails não lidos |
| `search_emails(query)` | Pesquisar por palavra-chave no assunto, corpo ou remetente |
| `filter_emails(...)` | Filtrar por destinatário e/ou intervalo de datas |
| `get_email(email_id)` | Recuperar um email específico pelo ID |
| `mark_email_as_read(id)` | Marcar um email como lido |
| `mark_email_as_unread(id)` | Marcar um email como não lido |
| `send_email(...)` | Enviar um novo email (simulado) |
| `delete_email(id)` | Excluir um email pelo ID |
| `search_unread_from_sender(addr)` | Retornar emails não lidos de um remetente específico (ex.: `boss@email.com`) |



In [ ]:
# Try with a request that may call an unavailable tool
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # list of tools that the LLM can access
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


#### 6.4.1 Tente novamente com `delete_email` habilitado

O que acontece se a ferramenta necessária não estiver disponível?

Na etapa anterior do laboratório, você concedeu ao LLM acesso à seguinte lista de ferramentas:
```python

tools=[
email_tools.search_unread_from_sender,

email_tools.list_unread_emails,

email_tools.search_emails,

email_tools.get_email,

email_tools.mark_email_as_read,

email_tools.send_email

]
```

E, devido às ferramentas disponíveis nessa lista, o agente não conseguiu excluir e-mails porque a ferramenta `delete_email` não estava disponível.

Agora, adicione `delete_email` à lista de ferramentas e execute a célula novamente. Desta vez, o agente terá tudo o que precisa para concluir a tarefa.

Dica: Observe a sequência de chamadas — após encontrar a mensagem desejada, o agente deve selecionar `delete_email` para concluir a ação.

In [ ]:
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


### 6.5. Ação direcionada: Excluir o e-mail “Happy Hour”

A caixa de entrada de teste vem pré-carregada com uma mensagem intitulada **“Happy Hour”**. Sua tarefa é instruir o agente a localizar essa mensagem e excluí-la.

Para referência, aqui está a entrada do conjunto de dados simulado:

```json
{
"id": 1,
"sender": "eric@work.com",
"recipient": "you@email.com",
"subject": "Happy Hour",
"body": "We're planning drinks this Friday!",
"timestamp": "2025-06-13T04:48:59.096908",
"read": false
}
````
Execute a próxima célula e observe como o agente pesquisa a mensagem e a exclui da caixa de entrada.

In [ ]:
prompt_ = build_prompt("Delete the happy hour email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

## 7. Conclusões Finais

- Neste laboratório não avaliado, **você explorou** como um agente de e-mail LLM interage com um serviço de e-mail simulado usando chamadas de ferramentas.

- **A chamada de ferramentas** permite que os LLMs vão além da geração de texto, possibilitando que eles chamem funções (ferramentas) e concluam tarefas de várias etapas.

- O conjunto de ferramentas disponíveis determina o que o agente pode e não pode fazer (por exemplo, sem `delete_email`, ele não pode remover mensagens).

- Docstrings claras e comportamento consistente ajudam o LLM a selecionar a ferramenta certa para cada etapa.

- O AISuite gerencia a camada de interação: expondo funções Python como ferramentas, aceitando parâmetros, fazendo solicitações de API e retornando resultados.

- Observar todo o fluxo de trabalho — desde o prompt → chamadas de ferramentas → saídas → resposta final — é fundamental para entender e aprimorar como os agentes raciocinam e agem.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>Parabéns!</strong>

Você acabou de guiar um agente assistente de e-mail com tecnologia LLM, transformando solicitações em linguagem natural em ações concretas em um serviço de e-mail simulado. Você viu como as ferramentas transformam a linguagem natural em operações confiáveis ​​— listar, pesquisar, marcar como lida, enviar e excluir — sem que você precise mexer nas requisições HTTP brutas.

Você aprendeu que as ferramentas que você expõe definem as capacidades reais do agente. Quando uma ferramenta está ausente, o agente pode raciocinar, mas não pode agir; Quando presente, ele pode encadear etapas, passar parâmetros e entregar exatamente o que você solicitou. Ao longo do processo, o AISuite cuidou da infraestrutura: exposição do esquema, vinculação de argumentos, execução e passagem de resultados — para que você pudesse se concentrar no que queria fazer, e não em como configurá-lo.

Com esse fluxo de trabalho em mãos, você está pronto para projetar agentes focados em tarefas que atuam com segurança e transparência, explicam o que fizeram e são escaláveis, desde simples solicitações até automação confiável de várias etapas em seus próprios serviços. 🌟
</div>